In [1]:
import sys

In [9]:
import h5py
import pandas as pd
import numpy as np
from datetime import datetime
import os

def optimized_read_hdf5(file_path, min_data_points=120000):
    # Configuración optimizada de tipos
    dtypes = {
        'value': np.float32,
        'sensor': 'category',
        'timestamp': 'datetime64[s]'
    }
    
    # Parseo preciso de la fecha desde el filename
    filename = os.path.basename(file_path)
    date_str = '_'.join(filename.split('_')[-3:]).split('.')[0]
    base_date = datetime.strptime(date_str, '%d_%m_%Y')
    
    dfs = []
    
    with h5py.File(file_path, 'r') as hdf_file:
        aventa_group = hdf_file['Aventa']
        
        for time_group in aventa_group.values():
            group_name = time_group.name.split('/')[-1]
            
            # Parseo robusto del timestamp
            try:
                h, m, s = map(int, group_name.split('_'))
                group_ts = base_date.replace(hour=h, minute=m, second=s)
            except:
                continue
            
            # Procesamiento vectorizado por sensor
            for sensor, node in time_group.items():
                if not isinstance(node, h5py.Group) or 'Value' not in node:
                    continue
                
                # Carga y transformación a 1D garantizada
                values = node['Value'][:].flatten()  # Asegura 1D
                
                if len(values) < min_data_points:
                    continue  # Filtro por cantidad mínima
                
                # Replicación eficiente de metadatos
                n_values = len(values)
                sensor_data = pd.Categorical(
                    [sensor] * n_values,
                    categories=[sensor],
                    ordered=False
                )
                
                timestamps = np.full(n_values, group_ts, dtype='datetime64[s]')
                
                # Construcción optimizada del DataFrame
                dfs.append(
                    pd.DataFrame({
                        'value': values.astype(dtypes['value']),
                        'sensor': sensor_data,
                        'timestamp': timestamps
                    })
                )
    
    # Concatenación final con tipos controlados
    final_df = pd.concat(dfs, ignore_index=True).astype(dtypes, copy=False)
    
    return final_df

# Uso:
df = optimized_read_hdf5("../../aventa_rotor_icing/Aventa_Taggenberg_01_11_2022.hdf5")
print(f"Memoria optimizada: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(df.head())

KeyboardInterrupt: 

In [12]:
import h5py
import pandas as pd
import numpy as np
import os

ruta_hdf5 = "../../aventa_rotor_icing/Aventa_Taggenberg_01_11_2022.hdf5"
dfs = []  # Lista para almacenar cada DataFrame generado
dataset_name = 'Aventa'
# Parseo preciso de la fecha desde el filename
filename = os.path.basename(ruta_hdf5)
date_str = '_'.join(filename.split('_')[-3:]).split('.')[0]
base_date = datetime.strptime(date_str, '%d_%m_%Y')

with h5py.File(ruta_hdf5, "r") as f:
    # Obtener la lista de timestamps disponibles en el dataset 'Aventa'
    time_names = list(f[dataset_name].keys())
    print(time_names)
    
    # Iterar sobre cada timestamp
    for time_stamp in time_names:
        data = {}
        # Filtrar los sensores que no comienzan con "Channel"
        sensor_names = [sensor for sensor in f[dataset_name][time_stamp].keys() if not sensor.startswith("Channel")]
        for sensor in sensor_names:
            # Leer los valores y convertir a float32
            sensor_values = f[dataset_name][time_stamp][sensor]['Value'][()]
            sensor_values = sensor_values.flatten().astype(np.float32)
            # Verificar si el sensor tiene suficientes datos
            if sensor_values.size > 120000:
                data[sensor] = sensor_values
        
        # Si se obtuvo algún dato, se crea el DataFrame correspondiente
        if data:
            df_temp = pd.DataFrame(data)
            # Insertar la columna timestamp
            df_temp.insert(0, "timestamp", time_stamp)
            df_temp.insert(0, "date", base_date)
            # Convertir la columna timestamp a tipo categórico para ahorrar memoria
            df_temp["timestamp"] = df_temp["timestamp"].astype("category")
            dfs.append(df_temp)

# Concatenar verticalmente todos los DataFrames generados
df_concatenado = pd.concat(dfs, axis=0, ignore_index=True)

print(df_concatenado)


['11_06_44', '11_16_45', '11_26_46', '11_36_47', '11_46_48', '11_56_49', '12_06_50', '12_16_51', '12_26_52', '12_30_22', '12_31_31', '12_42_04', '12_52_05', '13_02_06', '13_12_07', '13_22_08', '13_32_09', '13_42_10', '13_52_11', '14_02_12', '14_12_13', '14_22_14', '14_32_15', '14_42_16', '14_52_17']
              date timestamp  GEN_ACC_XX_01  GEN_ACC_YY_01  GEN_ACC_ZZ_01  \
0       2022-11-01  11_06_44       3.280851       2.258922       3.300865   
1       2022-11-01  11_06_44       3.280030       2.258951       3.305166   
2       2022-11-01  11_06_44       3.286086       2.286908       3.320716   
3       2022-11-01  11_06_44       3.285621       2.306745       3.316198   
4       2022-11-01  11_06_44       3.281640       2.291722       3.308524   
...            ...       ...            ...            ...            ...   
2645099 2022-11-01  14_52_17       3.291458       2.299684       3.322870   
2645100 2022-11-01  14_52_17       3.281359       2.293671       3.304140   
264510

In [2]:
(len(time_names)-3)*len(data['GEN_ACC_XX_01'])

2644664

In [3]:
len(data['GEN_ACC_XX_01'])

120212

In [4]:
df_concatenado["timestamp"].value_counts()

timestamp
12_26_52    120247
14_12_13    120246
13_32_09    120242
13_12_07    120240
11_36_47    120239
14_42_16    120239
11_56_49    120235
13_52_11    120235
12_52_05    120234
11_16_45    120232
14_22_14    120230
12_16_51    120230
11_26_46    120230
13_42_10    120229
12_06_50    120229
13_02_06    120229
14_32_15    120227
13_22_08    120226
14_02_12    120226
11_06_44    120224
11_46_48    120223
14_52_17    120212
Name: count, dtype: int64

In [5]:
df_concatenado["timestamp"].value_counts().sum()

np.int64(2645104)

In [6]:
print(f"Memoria usada: {df_concatenado.memory_usage(deep=True).sum() / 1e9:.2f} GB")

Memoria usada: 0.45 GB


In [7]:
df_concatenado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2645104 entries, 0 to 2645103
Data columns (total 29 columns):
 #   Column         Dtype  
---  ------         -----  
 0   timestamp      object 
 1   GEN_ACC_XX_01  float32
 2   GEN_ACC_YY_01  float32
 3   GEN_ACC_ZZ_01  float32
 4   L1_FBSG_01     float32
 5   L1_FBSG_02     float32
 6   L3_ACC_XX_01   float32
 7   L3_ACC_XX_02   float32
 8   L3_ACC_YY_01   float32
 9   L3_ACC_YY_02   float32
 10  L4_ACC_XX_01   float32
 11  L4_ACC_XX_02   float32
 12  L4_ACC_YY_01   float32
 13  L4_ACC_YY_02   float32
 14  L4_INC_XX_01   float32
 15  L4_INC_YY_01   float32
 16  L5_ACC_XX_01   float32
 17  L5_ACC_XX_02   float32
 18  L5_ACC_YY_01   float32
 19  L5_ACC_YY_02   float32
 20  L5_INC_XX_01   float32
 21  L5_INC_YY_01   float32
 22  MSB_ACC_XX_01  float32
 23  MSB_ACC_ZZ_02  float32
 24  MSH_ACC_XX_01  float32
 25  MSH_ACC_ZZ_01  float32
 26  NMF_ACC_XX_02  float32
 27  NMF_ACC_YY_01  float32
 28  NMF_ACC_YY_02  float32
dtypes: float32(28)

# Multiples Files

In [1]:
import h5py
import pandas as pd
import numpy as np
import os
from datetime import datetime

def process_file(file_path, dataset_name='Aventa', min_size=120000):
    """
    Procesa un único archivo HDF5 y retorna un DataFrame concatenado.
    
    Parámetros:
        file_path (str): Ruta al archivo HDF5.
        dataset_name (str): Nombre del dataset a extraer.
        min_size (int): Tamaño mínimo de datos requeridos por sensor.
    
    Retorna:
        pd.DataFrame: DataFrame con la información concatenada, incluyendo las columnas
                      'date' (extraída del nombre del archivo) y 'timestamp' (de cada grupo).
    """
    dfs = []
    # Parsear la fecha a partir del nombre del archivo
    filename = os.path.basename(file_path)
    # Se espera que el nombre tenga el formato: <algo>_<algo>_DD_MM_YYYY.hdf5
    date_str = '_'.join(filename.split('_')[-3:]).split('.')[0]
    base_date = datetime.strptime(date_str, '%d_%m_%Y')
    
    with h5py.File(file_path, "r") as f:
        # Obtener la lista de timestamps disponibles en el dataset
        time_names = list(f[dataset_name].keys())
        for time_stamp in time_names:
            data = {}
            # Filtrar los sensores que no comienzan con "Channel"
            sensor_names = [sensor for sensor in f[dataset_name][time_stamp].keys() if not sensor.startswith("Channel")]
            for sensor in sensor_names:
                # Leer los valores, aplanarlos y convertirlos a float32 para ahorrar memoria
                sensor_values = f[dataset_name][time_stamp][sensor]['Value'][()]
                sensor_values = sensor_values.flatten().astype(np.float32)
                if sensor_values.size > min_size:
                    data[sensor] = sensor_values
            if data:
                df_temp = pd.DataFrame(data)
                # Insertar la fecha y el timestamp en columnas
                df_temp.insert(0, "timestamp", time_stamp)
                base_date_np = np.datetime64(base_date, 'D')
                df_temp.insert(0, "date", base_date_np)
                # Convertir la columna timestamp a tipo categórico para optimizar memoria
                df_temp["timestamp"] = df_temp["timestamp"].astype("category")
                dfs.append(df_temp)
    
    if dfs:
        return pd.concat(dfs, axis=0, ignore_index=True)
    else:
        return pd.DataFrame()

def process_multiple_files(file_list, dataset_name='Aventa', min_size=120000):
    """
    Procesa una lista de archivos HDF5 y concatena los DataFrames resultantes en uno solo.
    
    Parámetros:
        file_list (list): Lista de rutas a archivos HDF5.
        dataset_name (str): Nombre del dataset a extraer.
        min_size (int): Tamaño mínimo de datos requeridos por sensor.
    
    Retorna:
        pd.DataFrame: DataFrame concatenado con la información de todos los archivos.
    """
    all_dfs = []
    for file_path in file_list:
        df = process_file(file_path, dataset_name, min_size)
        if not df.empty:
            all_dfs.append(df)
    if all_dfs:
        return pd.concat(all_dfs, axis=0, ignore_index=True)
    else:
        return pd.DataFrame()

# Ejemplo de uso:
file_list = ["../../aventa_rotor_icing/Aventa_Taggenberg_01_11_2022.hdf5",
              "../../aventa_rotor_icing/Aventa_Taggenberg_04_11_2022.hdf5"]
df_total = process_multiple_files(file_list)
print(df_total)


              date timestamp  GEN_ACC_XX_01  GEN_ACC_YY_01  GEN_ACC_ZZ_01  \
0       2022-11-01  11_06_44       3.280851       2.258922       3.300865   
1       2022-11-01  11_06_44       3.280030       2.258951       3.305166   
2       2022-11-01  11_06_44       3.286086       2.286908       3.320716   
3       2022-11-01  11_06_44       3.285621       2.306745       3.316198   
4       2022-11-01  11_06_44       3.281640       2.291722       3.308524   
...            ...       ...            ...            ...            ...   
4448576 2022-11-04  05_53_49       3.286097       2.272930       3.313828   
4448577 2022-11-04  05_53_49       3.283456       2.278262       3.312228   
4448578 2022-11-04  05_53_49       3.285578       2.284152       3.314505   
4448579 2022-11-04  05_53_49       3.286695       2.283043       3.315243   
4448580 2022-11-04  05_53_49       3.284840       2.284855       3.312379   

          L1_FBSG_01  L1_FBSG_02  L3_ACC_XX_01  L3_ACC_XX_02  L3_ACC_YY_01 

In [2]:
df_total["date"].value_counts()

date
2022-11-01    2645104
2022-11-04    1803477
Name: count, dtype: int64

In [3]:
df_total.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4448581 entries, 0 to 4448580
Data columns (total 30 columns):
 #   Column         Dtype        
---  ------         -----        
 0   date           datetime64[s]
 1   timestamp      object       
 2   GEN_ACC_XX_01  float32      
 3   GEN_ACC_YY_01  float32      
 4   GEN_ACC_ZZ_01  float32      
 5   L1_FBSG_01     float32      
 6   L1_FBSG_02     float32      
 7   L3_ACC_XX_01   float32      
 8   L3_ACC_XX_02   float32      
 9   L3_ACC_YY_01   float32      
 10  L3_ACC_YY_02   float32      
 11  L4_ACC_XX_01   float32      
 12  L4_ACC_XX_02   float32      
 13  L4_ACC_YY_01   float32      
 14  L4_ACC_YY_02   float32      
 15  L4_INC_XX_01   float32      
 16  L4_INC_YY_01   float32      
 17  L5_ACC_XX_01   float32      
 18  L5_ACC_XX_02   float32      
 19  L5_ACC_YY_01   float32      
 20  L5_ACC_YY_02   float32      
 21  L5_INC_XX_01   float32      
 22  L5_INC_YY_01   float32      
 23  MSB_ACC_XX_01  float32      
 24

In [4]:
import h5py
import pandas as pd
import numpy as np
import os
from datetime import datetime
import logging

# Configuración básica de logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

def process_file(file_path, dataset_name='Aventa', min_size=120000):
    """
    Procesa un único archivo HDF5 y retorna un DataFrame concatenado.
    
    Parámetros:
        file_path (str): Ruta al archivo HDF5.
        dataset_name (str): Nombre del dataset a extraer.
        min_size (int): Tamaño mínimo de datos requeridos por sensor.
    
    Retorna:
        pd.DataFrame: DataFrame con la información concatenada, incluyendo las columnas
                      'date' (extraída del nombre del archivo) y 'timestamp' (de cada grupo).
    """
    dfs = []
    # Parsear la fecha a partir del nombre del archivo
    filename = os.path.basename(file_path)
    # Se espera que el nombre tenga el formato: <algo>_<algo>_DD_MM_YYYY.hdf5
    date_str = '_'.join(filename.split('_')[-3:]).split('.')[0]
    base_date = datetime.strptime(date_str, '%d_%m_%Y')
    logging.info(f"Procesando archivo: {file_path} con fecha base: {base_date}")
    
    with h5py.File(file_path, "r") as f:
        # Obtener la lista de timestamps disponibles en el dataset
        time_names = list(f[dataset_name].keys())
        logging.info(f"Timestamps encontrados: {time_names}")
        
        # Iterar sobre cada timestamp
        for time_stamp in time_names:
            logging.info(f"Procesando timestamp: {time_stamp}")
            data = {}
            # Filtrar los sensores que no comienzan con "Channel"
            sensor_names = [sensor for sensor in f[dataset_name][time_stamp].keys() if not sensor.startswith("Channel")]
            for sensor in sensor_names:
                # Leer los valores, aplanarlos y convertirlos a float32 para ahorrar memoria
                sensor_values = f[dataset_name][time_stamp][sensor]['Value'][()]
                sensor_values = sensor_values.flatten().astype(np.float32)
                if sensor_values.size > min_size:
                    data[sensor] = sensor_values
                    logging.info(f"Sensor {sensor} procesado con {sensor_values.size} datos")
            if data:
                df_temp = pd.DataFrame(data)
                # Insertar la fecha y el timestamp en columnas
                df_temp.insert(0, "timestamp", time_stamp)
                base_date_np = np.datetime64(base_date, 'D')
                df_temp.insert(0, "date", base_date_np)
                # Convertir la columna timestamp a tipo categórico para optimizar memoria
                df_temp["timestamp"] = df_temp["timestamp"].astype("category")
                dfs.append(df_temp)
                logging.info(f"DataFrame para timestamp {time_stamp} creado con forma: {df_temp.shape}")
            else:
                logging.info(f"No se encontraron datos suficientes para timestamp: {time_stamp}")
    
    if dfs:
        df_result = pd.concat(dfs, axis=0, ignore_index=True)
        logging.info(f"Archivo {file_path} procesado. DataFrame final tiene forma: {df_result.shape}")
        return df_result
    else:
        logging.info(f"Archivo {file_path} procesado sin datos válidos.")
        return pd.DataFrame()

def process_multiple_files(file_list, dataset_name='Aventa', min_size=120000):
    """
    Procesa una lista de archivos HDF5 y concatena los DataFrames resultantes en uno solo.
    
    Parámetros:
        file_list (list): Lista de rutas a archivos HDF5.
        dataset_name (str): Nombre del dataset a extraer.
        min_size (int): Tamaño mínimo de datos requeridos por sensor.
    
    Retorna:
        pd.DataFrame: DataFrame concatenado con la información de todos los archivos.
    """
    all_dfs = []
    for file_path in file_list:
        logging.info(f"Iniciando procesamiento del archivo: {file_path}")
        df = process_file(file_path, dataset_name, min_size)
        if not df.empty:
            all_dfs.append(df)
        else:
            logging.info(f"No se obtuvieron datos válidos de {file_path}")
    if all_dfs:
        df_final = pd.concat(all_dfs, axis=0, ignore_index=True)
        logging.info(f"Procesamiento completo. DataFrame final tiene forma: {df_final.shape}")
        return df_final
    else:
        logging.info("No se procesó ningún archivo con datos válidos.")
        return pd.DataFrame()

# Ejemplo de uso:
file_list = ["../../aventa_rotor_icing/Aventa_Taggenberg_01_11_2022.hdf5",
             "../../aventa_rotor_icing/Aventa_Taggenberg_04_11_2022.hdf5"]
df_total = process_multiple_files(file_list)
print(df_total)


2025-03-25 15:39:58,305 - INFO - Iniciando procesamiento del archivo: ../../aventa_rotor_icing/Aventa_Taggenberg_01_11_2022.hdf5
2025-03-25 15:39:58,306 - INFO - Procesando archivo: ../../aventa_rotor_icing/Aventa_Taggenberg_01_11_2022.hdf5 con fecha base: 2022-11-01 00:00:00
2025-03-25 15:39:58,308 - INFO - Timestamps encontrados: ['11_06_44', '11_16_45', '11_26_46', '11_36_47', '11_46_48', '11_56_49', '12_06_50', '12_16_51', '12_26_52', '12_30_22', '12_31_31', '12_42_04', '12_52_05', '13_02_06', '13_12_07', '13_22_08', '13_32_09', '13_42_10', '13_52_11', '14_02_12', '14_12_13', '14_22_14', '14_32_15', '14_42_16', '14_52_17']
2025-03-25 15:39:58,309 - INFO - Procesando timestamp: 11_06_44
2025-03-25 15:39:58,313 - INFO - Sensor GEN_ACC_XX_01 procesado con 120224 datos
2025-03-25 15:39:58,314 - INFO - Sensor GEN_ACC_YY_01 procesado con 120224 datos
2025-03-25 15:39:58,316 - INFO - Sensor GEN_ACC_ZZ_01 procesado con 120224 datos
2025-03-25 15:39:58,318 - INFO - Sensor L1_FBSG_01 procesa

              date timestamp  GEN_ACC_XX_01  GEN_ACC_YY_01  GEN_ACC_ZZ_01  \
0       2022-11-01  11_06_44       3.280851       2.258922       3.300865   
1       2022-11-01  11_06_44       3.280030       2.258951       3.305166   
2       2022-11-01  11_06_44       3.286086       2.286908       3.320716   
3       2022-11-01  11_06_44       3.285621       2.306745       3.316198   
4       2022-11-01  11_06_44       3.281640       2.291722       3.308524   
...            ...       ...            ...            ...            ...   
4448576 2022-11-04  05_53_49       3.286097       2.272930       3.313828   
4448577 2022-11-04  05_53_49       3.283456       2.278262       3.312228   
4448578 2022-11-04  05_53_49       3.285578       2.284152       3.314505   
4448579 2022-11-04  05_53_49       3.286695       2.283043       3.315243   
4448580 2022-11-04  05_53_49       3.284840       2.284855       3.312379   

          L1_FBSG_01  L1_FBSG_02  L3_ACC_XX_01  L3_ACC_XX_02  L3_ACC_YY_01 